In [1]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  # standard alias
# from a pythonfile import a class or function
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder

from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.TipCantilever_30_20_20_midLoad import TipCantilever_30_20_20_midLoad
from problems.ThickenShell import ThickenShell

import pyvista as pv
try:
    pv.set_jupyter_backend("trame")
except Exception:
    pass

# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)
# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Code Directory: /home/arash/VoronoiImp-main
Test Step files Directory: /home/arash/VoronoiImp-main/Testparts
device: cuda


In [ ]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"

shape_path = str(FreeFormSurf1)
generator = CADTensorGenerator(
    deflection=0.001,
    angle=0.5,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=50,
    n_v=50,
    device=device,
)

mesh_df, faces_df, tensors = generator.generate_from_file(
    shape_path=shape_path,
    input_ring=1,
    visualize=True,
)

uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
faces_ijk = tensors["faces_ijk"]
pv_faces = tensors["pv_faces"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
min_vol_frac = tensors["min_vol_frac"]
bbox = tensors["BBX"]
print(f"pointxyz shape: {points_xyz.shape}")
print(f"Xu shape: {Xu.shape}")
print(f"Xv shape: {Xv.shape}")
print(f"faces_ijk shape: {faces_ijk.shape}")
print(f"face_id shape: {face_id.shape}")
print(f"boundary_idx_ring1 shape: {boundary_idx_ring1.shape}")
print(f"bbox: {bbox}")

viz.visualize_show_Model(points_xyz, pv_faces)


MinVolFrac: 0.05999999865889549
uv device: cuda:0
pointxyz shape: torch.Size([2550, 3])
Xu shape: torch.Size([2550, 3])
Xv shape: torch.Size([2550, 3])
faces_ijk shape: torch.Size([5000, 3])
face_id shape: torch.Size([2550])
boundary_idx_ring1 shape: torch.Size([200])
bbox: {0: {'xmin': -41.0, 'xmax': 41.0, 'ymin': -40.919095865559136, 'ymax': 40.919095865559136, 'zmin': 0.0, 'zmax': 100.0}}


Widget(value='<iframe src="http://localhost:35435/index.html?ui=P_0x797605fcc220_3&reconnect=auto" class="pyvi…

In [ ]:
shell_problem = ThickenShell(
    thickness=1.0,
    voxel_size=1.0,
    extra_layers=0,
    tensors=tensors,
    tangential_tol=1,
)

fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
shell_problem.debug_voxel_stats()
shell_problem.show_voxels_surface_and_bc()

In [ ]:


cfg = TrainingConfig(
    seed_number=5,
    use_anisotropy=False,
    target_volfrac=0.2,

    lam_fem=1.0,
    lam_vol=1.0,
    lam_rep=0.01,
    lam_bnd=0.01,

    lam_best_vol=5.0,
    lam_best_fem=0.0,

    comp_normalize_by=1e10,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=1000,
    context_vector_size=8,
    tau=0.02,
    beta=0.05,
    log_every=50,
    early_stop_start=200,
    patience=200,
    min_delta=1e-4,
)

trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    uv=uv,
    Xu=Xu,
    Xv=Xv,
    points_xyz=points_xyz,
    face_areas=face_areas,
    faces_ijk=faces_ijk,
    face_id=face_id,
    boundary_idx_ring1=boundary_idx_ring1,
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.01, show_solid=True)

In [ ]:
print("\n=== LAST FEM DEBUG ===")
for k, v in trainer.last_fem_debug.items():
    print(f"{k}: {v}")

In [ ]:
shell_problem = ThickenShell(
    thickness=1.0,
    voxel_size=0.1,
    extra_layers=0,
    tensors=tensors,
    tangential_tol=0.1,
)

fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
rho_surface = torch.full((points_xyz.shape[0],), 1, device=fem.device)
fiber_surface = torch.zeros(points_xyz.shape[0], 3, device=fem.device)
fiber_surface[:, 2] = 2.0

fem_fields = shell_problem.build_fem_fields_from_decoder(rho_surface, fiber_surface)

density = torch.tensor(fem_fields['density'], dtype=torch.float32, device=device)
phi = torch.tensor(fem_fields['phi'], dtype=torch.float32, device=device)
theta = torch.tensor(fem_fields['theta'], dtype=torch.float32, device=device)
print(phi)  
print(theta)
stress, comp = fem(density, phi, theta, penal=3)

print("stress:", float(stress))
print("comp  :", float(comp))


In [ ]:
phi = torch.zeros_like(density)
theta = torch.zeros_like(density)
_, comp_z = fem(density, phi, theta, penal=3)

phi = torch.zeros_like(density)
theta = torch.full_like(density, torch.pi / 2)
_, comp_x = fem(density, phi, theta, penal=3)

phi = torch.full_like(density, torch.pi / 2)
theta = torch.full_like(density, torch.pi / 2)
_, comp_y = fem(density, phi, theta, penal=3)

print("comp_x:", float(comp_x))
print("comp_y:", float(comp_y))
print("comp_z:", float(comp_z))

In [ ]:
valid = (shell_problem.elem_occupancy.reshape(-1) > 0) & (shell_problem.elem_sample_idx.reshape(-1) >= 0)
valid_t = torch.from_numpy(valid).to(device)

print("unique phi all   :", torch.unique(phi))
print("unique theta all :", torch.unique(theta))

print("unique phi valid   :", torch.unique(phi[valid_t]))
print("unique theta valid :", torch.unique(theta[valid_t]))

In [ ]:
sample_idx = shell_problem.elem_sample_idx.reshape(-1)
occ = shell_problem.elem_occupancy.reshape(-1)
valid = (occ > 0) & (sample_idx >= 0)

print("fiber_surface first 5:")
print(fiber_surface[:5].detach().cpu().numpy())

print("fiber_surface at valid samples:")
print(fiber_surface[sample_idx[valid][:5]].detach().cpu().numpy())

elem_density, elem_fiber = shell_problem.assign_surface_fields_to_voxels(rho_surface, fiber_surface)

print("elem_fiber valid first 5:")
print(elem_fiber[valid][:5])

print("unique valid elem_fiber:")
print(np.unique(np.round(elem_fiber[valid], 6), axis=0))